# AEDWIP Map

In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

# import joblib
# import math
import numpy as np
import os
import pandas as pd
# pd.set_option('display.max_rows', None)

# from sklearn.ensemble        import RandomForestClassifier

import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

import logging
# loglevel = "INFO"
loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

In [2]:
# setting the python path allows us to run python scripts from using
# the CLI. 
ORIG_PYTHONPATH = os.environ['PYTHONPATH']

#deconvolutionModules = notebookPath.parent.joinpath("../../deconvolutionAnalysis/python/")
# deconvolutionModules = notebookPath.parent.joinpath("../..")
deconvolutionModules = notebookPath.parent.joinpath("../../python")
print("deconvolutionModules: {}\n".format(deconvolutionModules))

PYTHONPATH = ORIG_PYTHONPATH + f':{deconvolutionModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../intraExtraRNA_POC/python/src")
# intraExtraRNA_POCModules=notebookPath.parent.joinpath("/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../../../../intraExtraRNA_POC/python/src")
intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../../intraExtraRNA_POC/python/src")
print("intraExtraRNA_POCModules: {}\n".format(intraExtraRNA_POCModules))

PYTHONPATH = PYTHONPATH + f':{intraExtraRNA_POCModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

os.environ["PYTHONPATH"] = PYTHONPATH
PYTHONPATH = os.environ["PYTHONPATH"]
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# to be able to import our local python files we need to set the sys.path
# https://stackoverflow.com/a/50155834
sys.path.append( str(deconvolutionModules) )
sys.path.append( str(intraExtraRNA_POCModules) )
print("\nsys.path:\n{}\n".format(sys.path))

deconvolutionModules: /private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../python

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../python

intraExtraRNA_POCModules: /private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../../intraExtraRNA_POC/python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../python:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../../intraExtraRNA_POC/python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../python:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../../intraExtraRNA_POC/python/src


sys.pat

In [3]:
# local imports
# from analysis.utilities import saveList
# from intraExtraRNA.elifeUtilities import loadElifeTrainingData
# from models.mlUtilities import saveLabelEncoder
from intraExtraRNA.elifeUtilities import selectFeatures

# Get Gene List

In [4]:
dataDir = '/private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20'
files = ['top20FromBest500GTEx_TCGA.LUAD_vs_all.results',
         'top20FromBest500GTEx_TCGA.LUSC_vs_all.results',
         'top20FromBest500GTEx_TCGA.Lung_vs_all.results',
         'top20FromBest500GTEx_TCGA.Whole_Blood_vs_all.results' ]

In [12]:
f = files[0]
category = f.split(".")[1]
category

'LUAD_vs_all'

In [17]:
def loadResults():
    retDict = dict()
    for f in files:
        category = f.split(".")[1]
        fPath = f'{dataDir}/{f}'
        df = pd.read_csv(fPath)
        print(f'loaded {fPath}')
        retDict[category] = df

    return retDict

deseqResultsDict = loadResults()

loaded /private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/top20FromBest500GTEx_TCGA.LUAD_vs_all.results
loaded /private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/top20FromBest500GTEx_TCGA.LUSC_vs_all.results
loaded /private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/top20FromBest500GTEx_TCGA.Lung_vs_all.results
loaded /private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/top20FromBest500GTEx_TCGA.Whole_Blood_vs_all.results


In [19]:
deseqResultsDict.keys()

dict_keys(['LUAD_vs_all', 'LUSC_vs_all', 'Lung_vs_all', 'Whole_Blood_vs_all'])

In [20]:
aedwipDF = deseqResultsDict['LUAD_vs_all']
aedwipDF

,name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
0,DES,89881.666167,-4.153133,0.180200,-23.047413,1.561467e-117,2.267881e-115
1,CLU,56070.788880,-2.273153,0.107877,-21.071755,1.444750e-98,1.413293e-96
2,KRT6A,36638.110998,-3.608496,0.236553,-15.254489,1.536931e-52,4.225783e-51
3,IGHA1,25987.995442,2.265104,0.173259,13.073548,4.663817e-39,7.564886e-38
4,CRYAB,21594.014674,-5.135958,0.127156,-40.390908,0.000000e+00,0.000000e+00
5,KRT17,19620.982687,-3.223883,0.188361,-17.115405,1.139206e-65,4.666331e-64
6,RGS5,18048.226869,-2.193606,0.118217,-18.555835,7.316229e-77,4.103218e-75
7,NDRG2,17790.572037,-2.249253,0.102673,-21.907055,2.225110e-106,2.565822e-104
8,SERPINA3,14604.186235,-2.506750,0.155628,-16.107276,2.267954e-58,7.491534e-57
9,SFTPB,14421.444640,6.019867,0.241647,24.911810,5.541549e-137,1.204986e-134


In [27]:
def getGeneSetDict(deseqResultsDict : pd.DataFrame) -> dict[str,pd.DataFrame] :
    retDict = dict()
    for key,resultsDF in deseqResultsDict.items():
        genesDF = resultsDF.loc[:, ['name']]
        retDict[key] = genesDF

    return retDict
        

geneSetDict = getGeneSetDict( deseqResultsDict)

In [28]:
geneSetDict['LUAD_vs_all']

,name
0,DES
1,CLU
2,KRT6A
3,IGHA1
4,CRYAB
5,KRT17
6,RGS5
7,NDRG2
8,SERPINA3
9,SFTPB
